# Harness Engineering — código de apoio

Material para acompanhar a mentoria **desenvolvendo de fato**. Cada seção
numerada espelha uma versão de código da apresentação; o código de apoio
(seção 0) define o mundo simulado que todas as seções usam.

**Como rodar:** execute as células de cima para baixo. Não é preciso chave de
API: `llm()` é uma simulação determinística que lê o histórico e propõe a
próxima ação, como um agente de negociação faria.

**O caso:** fatura em atraso de **R$ 2.400**; o cliente pede **30%** de
desconto; a política ilustrativa permite no máximo **15%** para este perfil.

| Seção | Versão | Slide |
|---|---|---|
| 0 | código de apoio (mundo + `llm()`) | — |
| 1 | Versão 0 — só uma chamada | 6 |
| 2 | Versão 1 — o loop | 9 |
| 3 | Versão 2 — ferramentas | 12 |
| 4 | Versão 3 — contexto | 18 |
| 5 | Regras determinísticas | 23 |
| 6 | Jev / decision model | 25 |
| 7 | Versão 4 — permission gate | 28 |
| 8 | Versão 5 — verificador | 34 |
| 9 | Pendência | 35 |
| 10 | Limites | 36 |
| 11 | Idempotência | 37 |
| 12 | Fallback | 38 |
| 13 | Trace | 39 |
| 14 | Evals | 41 |
| 15 | Versão final, parte A | 43 |
| 16 | Versão final, parte B | 44 |
| 17 | Fechamento: evals sobre o resultado | 41 |


## 0 · Código de apoio

O mundo em que o agente opera: um **CRM/financeiro mock** em memória, as
**ferramentas** (tools), e o **modelo simulado** (`llm`). Tudo o que as seções
numeradas usam é definido aqui.

> `llm(mensagens, ferramentas=None)` é a abstração de "chamar o modelo". Sem
> `ferramentas`, devolve só texto (Versão 0). Com `ferramentas`, age como um
> agente: confirma a fatura, registra a proposta respeitando o teto do
> contexto e encerra.


In [1]:
"""Ambiente simulado da mentoria — sem chave de API."""
import json
import re

# ---------------------------------------------------------------- constantes
TAREFA = "negociar_desconto_fatura"
USUARIO = {"id": 1, "nome": "atendente_1"}
CLIENTE_ID = 7
PEDIDO = "Tenho R$ 2.400 em atraso. Aceitam 30% de desconto?"
TETO = 0.15          # teto ilustrativo de desconto para este perfil
MAX_PASSOS = 5

# ------------------------------------------------------------------- mundo ---
class CRM:
    """Mock de CRM/financeiro: faturas, propostas e mensagens."""

    def __init__(self):
        self.faturas = {
            CLIENTE_ID: {"cliente_id": CLIENTE_ID, "valor": 2400.0,
                         "status": "em_atraso"},
        }
        self.propostas = {}   # id -> proposta
        self.chaves = {}      # idempotency_key -> id da proposta
        self.mensagens = []   # (cliente_id, texto)
        self._proxima_id = 1

    def fatura_do(self, cliente_id):
        return self.faturas.get(cliente_id)

    def registrar(self, cliente_id, desconto, chave=None):
        if chave is not None and chave in self.chaves:
            return self.chaves[chave]          # idempotente: mesma proposta
        pid = self._proxima_id
        self._proxima_id += 1
        self.propostas[pid] = {"id": pid, "cliente_id": cliente_id,
                               "desconto": desconto, "status": "registrada"}
        if chave is not None:
            self.chaves[chave] = pid
        return pid

    def proposta_por_chave(self, chave):
        pid = self.chaves.get(chave)
        return self.propostas.get(pid) if pid else None

    def enviar(self, cliente_id, texto):
        self.mensagens.append((cliente_id, texto))
        return {"status": "enviada"}


crm = CRM()


def reiniciar_mundo():
    """Recria o CRM (usado para começar cenários limpos)."""
    global crm
    crm = CRM()


# ------------------------------------------------------------------- tools ---
def consultar_fatura(cliente_id):
    """Lê a fatura atual do cliente (fonte de verdade)."""
    f = crm.fatura_do(cliente_id)
    if f is None:
        return {"tool": "consultar_fatura", "erro": "fatura_inexistente"}
    return {"tool": "consultar_fatura", **f}


def consultar_politica(perfil="fatura_em_atraso"):
    """Recupera a política de desconto do perfil."""
    return {"tool": "consultar_politica", "perfil": perfil,
            "desconto_maximo": TETO}


def consultar_cliente(cliente_id):
    """Recupera os dados do cliente (fonte de verdade)."""
    f = crm.fatura_do(cliente_id)
    if f is None:
        return None
    return {"cliente_id": cliente_id, "fatura": f["status"]}


def calcular_limite(cliente, politica):
    """Regra de negócio exata: o teto de desconto do perfil."""
    return politica.get("desconto_maximo", 0.0)


def registrar_proposta(valor, cliente, idempotency_key=None):
    """Registra a proposta. Enforce a regra de teto (determinística)."""
    teto = calcular_limite(cliente, consultar_politica())
    if valor > teto:
        return {"tool": "registrar_proposta",
                "erro": "ERRO_LIMITE_EXCEDIDO"}
    pid = crm.registrar(cliente, valor, chave=idempotency_key)
    return {"tool": "registrar_proposta", "proposta_id": pid,
            "desconto": valor, "status": "registrada"}


def enviar_mensagem(cliente_id, texto):
    """Comunica o cliente."""
    return {"tool": "enviar_mensagem", **crm.enviar(cliente_id, texto)}


FERRAMENTAS = [consultar_fatura, consultar_politica,
               registrar_proposta, enviar_mensagem]


In [2]:
# ------------------------------------------------------------- o modelo ---
class Resposta:
    """O que o modelo devolve: texto e, talvez, uma ação proposta."""

    def __init__(self, texto, acao=None, final=False):
        self.texto = texto
        self.acao = acao            # (nome_da_tool, argumentos) ou None
        self.final = final

    @property
    def tem_acao(self):
        return self.acao is not None

    def __repr__(self):
        return (f"Resposta(texto={self.texto!r}, acao={self.acao}, "
                f"final={self.final})")


def _observacoes(mensagens):
    return [m for m in mensagens
            if isinstance(m, dict) and m.get("tipo") == "observacao"]


def _teto_no_contexto(mensagens):
    """A simulação "lê" o teto formatado no contexto (desconto_maximo=X)."""
    for m in mensagens:
        if isinstance(m, dict):
            hit = re.search(r"desconto_maximo=(\d+\.\d+)",
                            m.get("content", ""))
            if hit:
                return float(hit.group(1))
    return None


def llm(mensagens, ferramentas=None):
    """Simulação determinística do modelo (sem chave de API).

    Sem `ferramentas`: devolve só texto (Versão 0).
    Com `ferramentas`: age como um agente de negociação —
      1. confirma a fatura atual;
      2. registra a proposta respeitando o teto do contexto
         (sem teto no contexto, propõe os 30% pedidos);
      3. encerra (final=True).
    """
    if ferramentas is None:
        return Resposta("Posso ajudar com a negociação. Qual o valor em atraso?")

    nomes = {f.__name__ for f in ferramentas}
    obs = _observacoes(mensagens)
    feitas = {m["tool"] for m in obs}

    if "consultar_fatura" in nomes and "consultar_fatura" not in feitas:
        return Resposta("Vou confirmar a fatura atual.",
                        acao=("consultar_fatura", {"cliente_id": CLIENTE_ID}))

    if "registrar_proposta" in nomes and "registrar_proposta" not in feitas:
        teto = _teto_no_contexto(mensagens)
        desconto = teto if teto is not None else 0.30
        return Resposta(
            f"Posso registrar uma proposta de {desconto:.0%} de desconto.",
            acao=("registrar_proposta",
                  {"valor": desconto, "cliente": CLIENTE_ID}))

    if "registrar_proposta" in feitas:
        ok = any(m.get("status") == "registrada" for m in obs
                 if m.get("tool") == "registrar_proposta")
        if ok:
            return Resposta("Proposta registrada.", final=True)
        return Resposta("A proposta não pôde ser registrada; vou escalar.",
                        final=True)

    return Resposta("Não sei qual o próximo passo.", final=True)


In [3]:
# --------------------------------------------------------------- helpers ---
INSTRUCOES_BASE = ("Você é um agente de negociação de faturas. Confirme os "
                   "dados antes de propor e respeite o teto de desconto.")


def formatar_tarefa(pedido, dados):
    """Monta o texto da tarefa com os dados recuperados."""
    pol = dados.get("politica") or {}
    cli = dados.get("cliente") or {}
    linhas = [f'Pedido: "{pedido}"']
    if cli:
        linhas.append(f"Cliente: {cli}")
    if pol:
        linhas.append(f"Política: desconto_maximo={pol['desconto_maximo']:.2f}")
    return "\n".join(linhas)


def montar_mensagens(tarefa, contexto=None, historico=None):
    """Monta a lista de mensagens (abstração de provider)."""
    msgs = [{"role": "system", "content": INSTRUCOES_BASE}]
    contexto = contexto or {}
    msgs.append({"role": "user", "content": formatar_tarefa(
        pedido=PEDIDO,
        dados={"cliente": contexto.get("cliente"),
               "politica": contexto.get("politica")},
    )})
    for item in (historico or []):
        if isinstance(item, Resposta):
            msgs.append({"role": "assistant", "content": item.texto})
        elif isinstance(item, dict):
            msgs.append({"role": "user",
                         "content": json.dumps(item, ensure_ascii=False),
                         "tipo": "observacao", "tool": item.get("tool")})
    return msgs


def buscar_politica_relevante(tarefa):
    """Recupera a política relevante para a tarefa (just-in-time)."""
    return consultar_politica()


def buscar_contexto_relevante(tarefa, estado, historico):
    """Recupera só o que a etapa atual precisa (just-in-time)."""
    return {"cliente": consultar_cliente(CLIENTE_ID),
            "politica": buscar_politica_relevante(tarefa)}


def executar(acao):
    """Interpreta a ação proposta e invoca a ferramenta correspondente."""
    nome, args = acao
    tool = {f.__name__: f for f in FERRAMENTAS}[nome]
    return tool(**args)


In [4]:
# --------------------------------------------- autoridade, idempotência, etc
def permissao_valida(usuario, acao):
    """A identidade do usuário pode acionar esta ferramenta?"""
    nome, _args = acao
    return nome in {f.__name__ for f in FERRAMENTAS}


def requer_aprovacao(acao):
    """Ações de alto impacto exigem aprovação humana antes de executar."""
    nome, args = acao
    return nome == "registrar_proposta" and args.get("valor", 0) >= 0.20


def chave_idempotente(tarefa, cliente_id):
    """Chave estável: repetir a mesma operação não duplica o efeito."""
    return f"{tarefa}:{cliente_id}"


def executar_com_idempotencia(acao):
    """Executa com proteção contra duplicidade nas ações que alteram estado."""
    nome, args = acao
    if nome == "registrar_proposta":
        chave = chave_idempotente("registrar", args.get("cliente", CLIENTE_ID))
        return registrar_proposta(args.get("valor", 0),
                                  args.get("cliente", CLIENTE_ID),
                                  idempotency_key=chave)
    return executar(acao)


def consultar_proposta_por_chave(chave):
    """Recupera o estado real após um timeout (não repetir cegamente)."""
    return crm.proposta_por_chave(chave)


# ------------------------------------------------------------------ trace ---
TRACES = []


def registrar_trace(etapa, acao=None, resultado=None, verificado=None):
    """Diário estruturado da execução (observabilidade)."""
    TRACES.append({"etapa": etapa, "acao": acao,
                   "resultado": resultado, "verificado": verificado})
    return TRACES[-1]


def ver_traces():
    for t in TRACES:
        print(f"[{t['etapa']}] acao={t['acao']} "
              f"resultado={t['resultado']} verificado={t['verificado']}")


# ------------------------------------------------------------- verificação ---
def consultar_sistema_de_registro():
    """Fonte de verdade: o que o CRM realmente contém."""
    return {"propostas": list(crm.propostas.values())}


def verificar_objetivo(tarefa, estado):
    """O objetivo foi atingido? (predicate sobre o mundo, não o texto)"""
    for p in estado.get("propostas", []):
        if p["status"] == "registrada" and p["desconto"] <= TETO:
            return True
    return False


def excedeu_tempo():
    return False


def excedeu_custo():
    return False


print("código de apoio carregado:",
      [f.__name__ for f in FERRAMENTAS])


código de apoio carregado: ['consultar_fatura', 'consultar_politica', 'registrar_proposta', 'enviar_mensagem']


## 1 · Versão 0 — só uma chamada ao modelo *(slide 6)*

O software envia informação e recebe informação. Sem ferramentas, a resposta
não tem efeito no mundo.


In [5]:
# Versão 0: só uma chamada ao modelo (slide 5)
reiniciar_mundo()
mensagens = [
    {"role": "user", "content": PEDIDO},
]
resposta = llm(mensagens)
print(resposta.texto)
print(f"CRM: {len(crm.propostas)} propostas, {len(crm.mensagens)} mensagens "
      "-> nenhum efeito no mundo")


Posso ajudar com a negociação. Qual o valor em atraso?
CRM: 0 propostas, 0 mensagens -> nenhum efeito no mundo


## 2 · Versão 1 — o loop *(slide 9)*

Agora o agente pode dar vários passos: o modelo propõe a chamada, o software
interpreta e decide executar. Ainda **sem contexto**, o modelo propõe os 30%
pedidos — e o executor rejeita (a regra de teto já existe no mundo).


In [6]:
# Versão 1: o agente pode dar vários passos (slide 8)
reiniciar_mundo()

def executar_agente_v1(tarefa, usuario):
    historico = []
    for passo in range(MAX_PASSOS):
        mensagens = montar_mensagens(tarefa, None, historico)
        resposta = llm(mensagens, ferramentas=FERRAMENTAS)
        if resposta.final:
            return resposta.texto
        observacao = executar(resposta.acao)
        historico += [resposta, observacao]
        print(f"passo {passo + 1}: {resposta.texto}")
        print(f"         -> {observacao}")
    return "ESCALAR"


print("resultado:", executar_agente_v1(TAREFA, USUARIO))
print("propostas no CRM:", len(crm.propostas))


passo 1: Vou confirmar a fatura atual.
         -> {'tool': 'consultar_fatura', 'cliente_id': 7, 'valor': 2400.0, 'status': 'em_atraso'}
passo 2: Posso registrar uma proposta de 30% de desconto.
         -> {'tool': 'registrar_proposta', 'erro': 'ERRO_LIMITE_EXCEDIDO'}
resultado: A proposta não pôde ser registrada; vou escalar.
propostas no CRM: 0


## 3 · Versão 2 — registramos as tools *(slide 12)*

Quem registra capacidades é a aplicação; quem propõe usar uma delas é o
modelo; quem executa é o harness. Expor a ferramenta e permitir a execução
são controles distintos.


In [7]:
# Versão 2: registramos as tools (slide 11)
reiniciar_mundo()
print("ferramentas = [", ", ".join(f.__name__ for f in FERRAMENTAS), "]")

mensagens = montar_mensagens(TAREFA, None, [])
resposta = llm(mensagens=mensagens, ferramentas=FERRAMENTAS)
print("o modelo propõe:", resposta.acao)
print("a tool call é uma intenção — ainda não mudou nada no CRM")


ferramentas = [ consultar_fatura, consultar_politica, registrar_proposta, enviar_mensagem ]
o modelo propõe: ('consultar_fatura', {'cliente_id': 7})
a tool call é uma intenção — ainda não mudou nada no CRM


## 4 · Versão 3 — o harness monta as mensagens *(slide 18)*

O contexto não entra sozinho: nós buscamos e formatamos. Com o teto no
contexto, o modelo propõe 15% — não os 30% pedidos.


In [8]:
# Versão 3: o harness monta as mensagens (slide 17)
reiniciar_mundo()
cliente = consultar_cliente(CLIENTE_ID)
politica = buscar_politica_relevante(TAREFA)
contexto = {"cliente": cliente, "politica": politica}

mensagens = montar_mensagens(TAREFA, contexto, [])
print("input montado:")
print(mensagens[1]["content"])
print()

# passo 1: o modelo confirma a fatura
resposta = llm(mensagens, ferramentas=FERRAMENTAS)
observacao = executar(resposta.acao)
print("passo 1 ->", observacao)

# passo 2: com o teto no contexto, propõe a proposta certa
mensagens2 = montar_mensagens(TAREFA, contexto, [resposta, observacao])
resposta2 = llm(mensagens2, ferramentas=FERRAMENTAS)
print("passo 2 -> o modelo propõe:", resposta2.acao)


input montado:
Pedido: "Tenho R$ 2.400 em atraso. Aceitam 30% de desconto?"
Cliente: {'cliente_id': 7, 'fatura': 'em_atraso'}
Política: desconto_maximo=0.15

passo 1 -> {'tool': 'consultar_fatura', 'cliente_id': 7, 'valor': 2400.0, 'status': 'em_atraso'}
passo 2 -> o modelo propõe: ('registrar_proposta', {'valor': 0.15, 'cliente': 7})


## 5 · Regras determinísticas continuam valiosas *(slide 23)*

O modelo pode explicar a proposta; o código calcula e impõe o teto. Dizer o
limite melhora a escolha; validar no executor impede o efeito.


In [9]:
# Regras determinísticas (slide 22)
reiniciar_mundo()
teto = calcular_limite(cliente, politica)
contexto["desconto_maximo"] = teto
print("teto calculado pelo código:", f"{teto:.0%}")

print("registrar 30% (pedido) ->",
      registrar_proposta(0.30, CLIENTE_ID))
print("registrar 15% (teto)   ->",
      registrar_proposta(0.15, CLIENTE_ID))


teto calculado pelo código: 15%
registrar 30% (pedido) -> {'tool': 'registrar_proposta', 'erro': 'ERRO_LIMITE_EXCEDIDO'}
registrar 15% (teto)   -> {'tool': 'registrar_proposta', 'proposta_id': 1, 'desconto': 0.15, 'status': 'registrada'}


## 6 · Exemplo: Jev para uma decisão delimitada *(slide 25)*

Um decision model recebe estado e uma pergunta focada e devolve uma saída
tipada. A probabilidade orienta a estratégia; o LLM formula a mensagem.
*(Exemplo, não requisito — e probabilidade prevista não é calibração
garantida.)*


In [10]:
# Jev: decisão delimitada (slide 24)
def prever_probabilidade_pagamento(estado_cliente):
    """Simulação de um decision model (ex.: Jev). Em produção: modelo
    treinado; aqui, uma heurística ilustrativa."""
    return 0.22

estado_cliente = {"cliente_id": CLIENTE_ID, "fatura": "em_atraso"}
p = prever_probabilidade_pagamento(estado_cliente)
estrategia = (
    "negociacao_flexivel" if p < 0.30
    else "negociacao_padrao"
)
contexto_jev = {"estrategia": estrategia}
print(f"probabilidade = {p:.2f}  ->  estratégia = {estrategia}")
print("o LLM usa essa estratégia na conversa; a decisão é tipada")


probabilidade = 0.22  ->  estratégia = negociacao_flexivel
o LLM usa essa estratégia na conversa; a decisão é tipada


## 7 · Versão 4 — permission gate no executor *(slide 28)*

Capacidade não é autoridade. Entre a proposta e a execução há um gate que
considera identidade, cliente alvo, operação, valor e contexto.


In [11]:
# Versão 4: permission gate no executor (slide 27)
def obter_acao(resposta):
    """Extrai a ação proposta de uma Resposta."""
    return resposta.acao


def executar_com_gate(usuario, resposta):
    acao = obter_acao(resposta)
    if not permissao_valida(usuario, acao):
        return "NEGADO"
    if requer_aprovacao(acao):
        return "AGUARDANDO_APROVACAO"
    return executar(acao)


reiniciar_mundo()
r15 = Resposta("proposta de 15%",
               acao=("registrar_proposta",
                     {"valor": 0.15, "cliente": CLIENTE_ID}))
r25 = Resposta("proposta de 25%",
               acao=("registrar_proposta",
                     {"valor": 0.25, "cliente": CLIENTE_ID}))
print("15% ->", executar_com_gate(USUARIO, r15))
print("25% ->", executar_com_gate(USUARIO, r25))


15% -> {'tool': 'registrar_proposta', 'proposta_id': 1, 'desconto': 0.15, 'status': 'registrada'}
25% -> AGUARDANDO_APROVACAO


## 8 · Versão 5 — um verificador separado *(slide 34)*

A tool call é uma tentativa; o estado externo confirma (ou não) o outcome. O
verificador consulta a fonte de verdade, separado de quem gera a resposta.


In [12]:
# Versão 5: um verificador separado (slide 33)
reiniciar_mundo()
acao = ("registrar_proposta", {"valor": 0.15, "cliente": CLIENTE_ID})
resultado = executar(acao)
print("resultado da tool:", resultado)

estado_real = consultar_sistema_de_registro()
confirmado = verificar_objetivo(tarefa=TAREFA, estado=estado_real)
print("verificador confirma o objetivo?", confirmado)
print("propostas no CRM:", estado_real["propostas"])


resultado da tool: {'tool': 'registrar_proposta', 'proposta_id': 1, 'desconto': 0.15, 'status': 'registrada'}
verificador confirma o objetivo? True
propostas no CRM: [{'id': 1, 'cliente_id': 7, 'desconto': 0.15, 'status': 'registrada'}]


## 9 · Sucesso, pendência e falha não são a mesma coisa *(slide 35)*

Aceitar a solicitação não é concluir a mudança. Se a atualização é
assíncrona, a verificação imediata ainda não encontra o novo status — e o
harness expressa **pendência**, não sucesso nem falha.


In [13]:
# Pendência (slide 34)
reiniciar_mundo()
crm.registrar(CLIENTE_ID, 0.15)          # a escrita aconteceu
# ...mas o sistema de registro (outra fonte) ainda não propagou:
estado_real = {"propostas": []}
print("verificação imediata:", verificar_objetivo(TAREFA, estado_real))
print("-> PENDENTE: reconsultar / aguardar evento (não é falha, não é sucesso)")


verificação imediata: False
-> PENDENTE: reconsultar / aguardar evento (não é falha, não é sucesso)


## 10 · Um agente precisa saber parar *(slide 36)*

Limites de passos, tempo e custo são **políticas do processo**, não
instruções do modelo. Poder sem limites vira risco operacional.


In [14]:
# Limites (slide 35)
for passo in range(MAX_PASSOS):
    if excedeu_tempo() or excedeu_custo():
        print("-> ESCALAR (limite de tempo/custo)")
        break
    print(f"passo {passo + 1} dentro dos limites")
else:
    print(f"-> sem sucesso em {MAX_PASSOS} passos: ESCALAR")


passo 1 dentro dos limites
passo 2 dentro dos limites
passo 3 dentro dos limites
passo 4 dentro dos limites
passo 5 dentro dos limites
-> sem sucesso em 5 passos: ESCALAR


## 11 · Retry sem idempotência pode duplicar ações *(slide 37)*

Timeout diz que perdemos o retorno — não que a operação não aconteceu. Retry
seguro precisa considerar duplicidade: a mesma chave não gera duas propostas.


In [15]:
# Idempotência (slide 36)
reiniciar_mundo()
chave = chave_idempotente(TAREFA, CLIENTE_ID)
print("chave:", chave)

r1 = registrar_proposta(0.15, CLIENTE_ID, idempotency_key=chave)
print("1ª tentativa:", r1)

# após um timeout, a 2ª tentativa usa a mesma chave:
r2 = registrar_proposta(0.15, CLIENTE_ID, idempotency_key=chave)
print("2ª tentativa:", r2)
print("propostas no CRM:", len(crm.propostas), "(deveria ser 1)")


chave: negociar_desconto_fatura:7
1ª tentativa: {'tool': 'registrar_proposta', 'proposta_id': 1, 'desconto': 0.15, 'status': 'registrada'}
2ª tentativa: {'tool': 'registrar_proposta', 'proposta_id': 1, 'desconto': 0.15, 'status': 'registrada'}
propostas no CRM: 1 (deveria ser 1)


## 12 · Fallback e supervisão humana *(slide 38)*

Autonomia não é tudo ou nada. O agente pode parar, pedir confirmação ou
escalar — e cada saída tem um tratamento diferente.


In [16]:
# Fallback e supervisão humana (slide 37)
def decidir_saida(acao, resultado):
    if resultado in ("NEGADO", "AGUARDANDO_APROVACAO"):
        return resultado
    if isinstance(resultado, dict) and resultado.get("erro"):
        return "ERRO_IRRECUPERAVEL"
    return "SUCESSO"


acao = ("registrar_proposta", {"valor": 0.15, "cliente": CLIENTE_ID})
print("aprovacao: ", decidir_saida(acao, "AGUARDANDO_APROVACAO"))
print("erro:      ", decidir_saida(acao,
      {"tool": "registrar_proposta", "erro": "ERRO_LIMITE_EXCEDIDO"}))
print("sucesso:   ", decidir_saida(acao,
      {"tool": "registrar_proposta", "proposta_id": 1, "status": "registrada"}))


aprovacao:  AGUARDANDO_APROVACAO
erro:       ERRO_IRRECUPERAVEL
sucesso:    SUCESSO


## 13 · Observabilidade: qual caminho levou ao resultado? *(slide 39)*

O trace é o diário estruturado da execução. Sem trajetória, temos somente o
palpite sobre a causa.


In [17]:
# Observabilidade (slide 38)
reiniciar_mundo()
TRACES.clear()
acao = ("registrar_proposta", {"valor": 0.15, "cliente": CLIENTE_ID})
resultado = executar(acao)
registrar_trace(
    etapa="registrar_proposta",
    acao=acao,
    resultado=resultado,
    verificado=True,
)
ver_traces()


[registrar_proposta] acao=('registrar_proposta', {'valor': 0.15, 'cliente': 7}) resultado={'tool': 'registrar_proposta', 'proposta_id': 1, 'desconto': 0.15, 'status': 'registrada'} verificado=True


## 14 · Evals: medir o que importa *(slide 41)*

Regras exatas → **assertions**; objetivo de negócio → **verificadores**;
qualidade subjetiva → **rubrica**. Não use LLM-juiz para checar soma exata
quando o código faz melhor.


In [18]:
# Evals (slide 40)
reiniciar_mundo()
registrar_proposta(0.15, CLIENTE_ID)


def eval_teto():
    """Assertion determinística: nenhuma proposta acima do teto."""
    return all(p["desconto"] <= TETO
               for p in consultar_sistema_de_registro()["propostas"])


def eval_outcome():
    """Verificador de outcome: a proposta existe no mundo."""
    return verificar_objetivo(TAREFA, consultar_sistema_de_registro())


print("assertion (nenhuma acima do teto):", eval_teto())
print("verificador (proposta registrada):", eval_outcome())


assertion (nenhuma acima do teto): True
verificador (proposta registrada): True


## 15 · Versão final, parte A — preparar e propor *(slide 43)*

Carregar estado, montar contexto, montar input, expor ferramentas, chamar o
modelo. As linhas são **decisões do desenvolvedor**, não “magia” da API.


In [19]:
# Versão final, parte A (slide 42)
def carregar_estado(usuario):
    return {"cliente_id": CLIENTE_ID, "fatura": "em_atraso"}


def ferramentas_disponiveis(usuario, estado):
    """O harness escolhe quais capacidades expor neste estado."""
    return FERRAMENTAS


def preparar_passo(tarefa, usuario, estado, historico):
    """Parte A: preparar e propor (um passo do loop)."""
    contexto = buscar_contexto_relevante(tarefa, estado, historico)
    mensagens = montar_mensagens(tarefa, contexto, historico)
    ferramentas = ferramentas_disponiveis(usuario, estado)
    resposta = llm(mensagens=mensagens, ferramentas=ferramentas)
    return contexto, mensagens, ferramentas, resposta


reiniciar_mundo()
estado = carregar_estado(USUARIO)
contexto, mensagens, ferramentas, resposta = preparar_passo(
    TAREFA, USUARIO, estado, [])
print("contexto:    ", contexto)
print("ferramentas: ", [f.__name__ for f in ferramentas])
print("resposta:    ", resposta.texto, "| acao:", resposta.acao)


contexto:     {'cliente': {'cliente_id': 7, 'fatura': 'em_atraso'}, 'politica': {'tool': 'consultar_politica', 'perfil': 'fatura_em_atraso', 'desconto_maximo': 0.15}}
ferramentas:  ['consultar_fatura', 'consultar_politica', 'registrar_proposta', 'enviar_mensagem']
resposta:     Vou confirmar a fatura atual. | acao: ('consultar_fatura', {'cliente_id': 7})


## 16 · Versão final, parte B — autorizar, observar, verificar *(slide 44)*

Tudo que impede uma sugestão de virar um efeito indevido: validar autoridade,
executar com idempotência, observar, registrar a trajetória e verificar o
resultado. Sem proposta real no CRM, o sistema não pode retornar sucesso.


In [20]:
# Versão final, parte B (slide 43)
def registrar_no_historico(historico, item):
    historico.append(item)


def extrair_acao(resposta):
    return resposta.acao


def autorizado(usuario, acao):
    return permissao_valida(usuario, acao) and not requer_aprovacao(acao)


def observar(resultado):
    return resultado


def recarregar_estado_real():
    return {"propostas": list(crm.propostas.values())}


def executar_agente(tarefa, usuario):
    historico = []
    estado = carregar_estado(usuario)
    for passo in range(MAX_PASSOS):
        if excedeu_tempo() or excedeu_custo():
            return "ESCALAR"

        # ---- parte A: preparar e propor ----
        contexto, mensagens, ferramentas, resposta = preparar_passo(
            tarefa, usuario, estado, historico)
        registrar_no_historico(historico, resposta)

        # ---- parte B: autorizar, observar, verificar ----
        if resposta.tem_acao:
            acao = extrair_acao(resposta)
            if not autorizado(usuario, acao):
                return "ESCALAR_OU_NEGAR"

            resultado = executar_com_idempotencia(acao)
            observacao = observar(resultado)
            registrar_no_historico(historico, observacao)
            estado = recarregar_estado_real()
            confirmado = verificar_objetivo(tarefa, estado)
            registrar_trace(acao[0], acao, observacao, confirmado)
        else:
            confirmado = verificar_objetivo(tarefa, estado)

        if confirmado:
            return "SUCESSO"
    return "ESCALAR"


reiniciar_mundo()
TRACES.clear()
print("resultado:", executar_agente(TAREFA, USUARIO))
print()
ver_traces()


resultado: SUCESSO

[consultar_fatura] acao=('consultar_fatura', {'cliente_id': 7}) resultado={'tool': 'consultar_fatura', 'cliente_id': 7, 'valor': 2400.0, 'status': 'em_atraso'} verificado=False
[registrar_proposta] acao=('registrar_proposta', {'valor': 0.15, 'cliente': 7}) resultado={'tool': 'registrar_proposta', 'proposta_id': 1, 'desconto': 0.15, 'status': 'registrada'} verificado=True


## 17 · Fechamento — evals sobre o resultado final *(slide 41)*

Os mesmos evals da seção 14, agora sobre o que o agente final realmente
produziu no mundo.


In [21]:
# Fechamento: evals sobre o resultado final (slide 40)
print("assertion (nenhuma acima do teto):",
      all(p["desconto"] <= TETO for p in crm.propostas.values()))
print("verificador (proposta registrada):",
      verificar_objetivo(TAREFA, consultar_sistema_de_registro()))
print("propostas no CRM:", list(crm.propostas.values()))
print("mensagens ao cliente:", crm.mensagens)


assertion (nenhuma acima do teto): True
verificador (proposta registrada): True
propostas no CRM: [{'id': 1, 'cliente_id': 7, 'desconto': 0.15, 'status': 'registrada'}]
mensagens ao cliente: []
